In [2]:
import sqlite3
import pandas as pd

# Load superstore data
df = pd.read_csv(r'E:\data-analyst-portfolio\Sample-Superstore.csv', encoding='latin1')

# Create a SQLite database in memory
conn = sqlite3.connect(':memory:')

# Load dataframe into SQL table
df.to_sql('superstore', conn, index=False, if_exists='replace')

# Now write SQL queries on your Superstore data
query1 = """
SELECT Region, 
       SUM(Sales) as Total_Sales,
       SUM(Profit) as Total_Profit,
       COUNT(*) as Total_Orders
FROM superstore
GROUP BY Region
ORDER BY Total_Sales DESC
"""
result1 = pd.read_sql_query(query1, conn)
print("Sales by Region:\n", result1)

query2 = """
SELECT Category,
       AVG(Profit) as Avg_Profit,
       SUM(Sales) as Total_Sales
FROM superstore
GROUP BY Category
ORDER BY Avg_Profit DESC
"""
result2 = pd.read_sql_query(query2, conn)
print("\nProfit by Category:\n", result2)

query3 = """
SELECT `Sub-Category`,
       SUM(Sales) as Total_Sales,
       SUM(Profit) as Total_Profit
FROM superstore
GROUP BY `Sub-Category`
ORDER BY Total_Sales DESC
LIMIT 10
"""
result3 = pd.read_sql_query(query3, conn)
print("\nTop 10 Sub-Categories:\n", result3)

Sales by Region:
     Region  Total_Sales  Total_Profit  Total_Orders
0     West  725457.8245   108418.4489          3203
1     East  678781.2400    91522.7800          2848
2  Central  501239.8908    39706.3625          2323
3    South  391721.9050    46749.4303          1620

Profit by Category:
           Category  Avg_Profit  Total_Sales
0       Technology   78.752002  836154.0330
1  Office Supplies   20.327050  719047.0320
2        Furniture    8.699327  741999.7953

Top 10 Sub-Categories:
   Sub-Category  Total_Sales  Total_Profit
0       Phones  330007.0540    44515.7306
1       Chairs  328449.1030    26590.1663
2      Storage  223843.6080    21278.8264
3       Tables  206965.5320   -17725.4811
4      Binders  203412.7330    30221.7633
5     Machines  189238.6310     3384.7569
6  Accessories  167380.3180    41936.6357
7      Copiers  149528.0300    55617.8249
8    Bookcases  114879.9963    -3472.5560
9   Appliances  107532.1610    18138.0054


Query 4 — Find orders with negative profit:

In [3]:
query4 = """
SELECT Region, Category, `Sub-Category`,
       Sales, Profit,
       ROUND((Profit/Sales)*100, 2) as Profit_Margin_Pct
FROM superstore
WHERE Profit < 0
ORDER BY Profit ASC
LIMIT 15
"""
result4 = pd.read_sql_query(query4, conn)
print("Most Loss-Making Orders:\n", result4)

Most Loss-Making Orders:
      Region         Category Sub-Category      Sales     Profit  \
0      East       Technology     Machines   4499.985 -6599.9780   
1     South       Technology     Machines   7999.980 -3839.9904   
2   Central  Office Supplies      Binders   2177.584 -3701.8928   
3      West       Technology     Machines   2549.985 -3399.9800   
4   Central  Office Supplies      Binders   1889.990 -2929.4845   
5      East       Technology     Machines   1799.994 -2639.9912   
6   Central  Office Supplies      Binders   1525.188 -2287.7820   
7     South        Furniture       Tables   4297.644 -1862.3124   
8   Central  Office Supplies      Binders   1088.792 -1850.9464   
9     South       Technology     Machines  22638.480 -1811.0784   
10     East        Furniture    Bookcases   3083.430 -1665.0522   
11  Central  Office Supplies      Binders    896.990 -1480.0335   
12  Central       Technology     Machines   8159.952 -1359.9920   
13    South  Office Supplies      Bi

Query 5 — Which state has highest sales:

In [4]:
query5 = """
SELECT State,
       SUM(Sales) as Total_Sales,
       SUM(Profit) as Total_Profit,
       COUNT(*) as Orders
FROM superstore
GROUP BY State
ORDER BY Total_Sales DESC
LIMIT 10
"""
result5 = pd.read_sql_query(query5, conn)
print("Top 10 States by Sales:\n", result5)

Top 10 States by Sales:
           State  Total_Sales  Total_Profit  Orders
0    California  457687.6315    76381.3871    2001
1      New York  310876.2710    74038.5486    1128
2         Texas  170188.0458   -25729.3563     985
3    Washington  138641.2700    33402.6517     506
4  Pennsylvania  116511.9140   -15559.9603     587
5       Florida   89473.7080    -3399.3017     383
6      Illinois   80166.1010   -12607.8870     492
7          Ohio   78258.1360   -16971.3766     469
8      Michigan   76269.6140    24463.1876     255
9      Virginia   70636.7200    18597.9504     224


Query 6 — Customer segment analysis:

In [5]:
query6 = """
SELECT Segment,
       COUNT(*) as Total_Orders,
       SUM(Sales) as Total_Sales,
       AVG(Sales) as Avg_Order_Value,
       SUM(Profit) as Total_Profit
FROM superstore
GROUP BY Segment
ORDER BY Total_Sales DESC
"""
result6 = pd.read_sql_query(query6, conn)
print("Segment Analysis:\n", result6)

Segment Analysis:
        Segment  Total_Orders   Total_Sales  Avg_Order_Value  Total_Profit
0     Consumer          5191  1.161401e+06       223.733644   134119.2092
1    Corporate          3020  7.061464e+05       233.823300    91979.1340
2  Home Office          1783  4.296531e+05       240.972041    60298.6785


Query 7 — Discount impact on profit:

In [6]:
query7 = """
SELECT 
    CASE 
        WHEN Discount = 0 THEN 'No Discount'
        WHEN Discount <= 0.2 THEN 'Low Discount'
        WHEN Discount <= 0.4 THEN 'Medium Discount'
        ELSE 'High Discount'
    END as Discount_Level,
    COUNT(*) as Orders,
    AVG(Profit) as Avg_Profit,
    SUM(Sales) as Total_Sales
FROM superstore
GROUP BY Discount_Level
ORDER BY Avg_Profit DESC
"""
result7 = pd.read_sql_query(query7, conn)
print("Discount Impact on Profit:\n", result7)

Discount Impact on Profit:
     Discount_Level  Orders  Avg_Profit   Total_Sales
0      No Discount    4798   66.900292  1.087908e+06
1     Low Discount    3803   26.501571  8.465222e+05
2  Medium Discount     460  -77.864055  2.341379e+05
3    High Discount     933 -106.708028  1.286323e+05
